In [ ]:
import sympy as sp
from IPython.display import display

# ==============================================================================
# 1. SymPy Settings and Symbol Definitions
# ==============================================================================

sp.init_printing(use_unicode=True)

z = sp.Symbol('z', complex=True)
n = sp.Symbol('n', integer=True, nonnegative=True)

print("=== METHOD: Complex Integration with Symbolic Pole Detection & Clean Expansion ===")

# 1. Define raw X(z) and convert to exact rational fractions
X_z_raw = z / ((z + 0.2) * (z**2 - 0.4 * z + 0.04))
X_z = sp.nsimplify(X_z_raw)

print("\n1. Function X(z) strictly as exact fractions:")
display(X_z)

# 2. Auxiliary function Y(z) = (X(z) / z) * z^n
Y_z = sp.simplify((X_z / z) * z**n)

print("\n2. Auxiliary function Y(z):")
display(Y_z)

# 3. Extract poles and their multiplicities symbolically
Y_z_base = X_z / z
num_y, den_y = sp.fraction(sp.cancel(Y_z_base))

# Use factor_list to get poles and multiplicities cleanly
factored_den = sp.factor_list(den_y)
print("\n3. Pole analysis (factors and multiplicities):")
display(factored_den)

# Extract poles dynamically from the factors list (ignoring z = 0 if any)
poles_info = []
for factor, mult in factored_den[1]:
    roots = sp.solve(factor, z)
    for root in roots:
        if root != 0:
            poles_info.append((root, mult))

print("\nDetected non-zero poles and multiplicities:")
display(poles_info)

# 4. Calculate residues and terms for each pole
res_sum = 0
u = sp.Heaviside(n)

print("\n4. Calculating individual residue terms:")
for p_exact, mult in poles_info:
    if mult == 1:
        # Simple pole residue formula
        residue = sp.limit((z - p_exact) * Y_z, z, p_exact)
        term = sp.simplify(residue * u)
        print(f" -> Simple Pole at z = {p_exact}:")
    elif mult > 1:
        # Multiple pole residue formula: 1/(m-1)! * lim_{z->p} d^(m-1)/dz^(m-1) [ (z-p)^m * Y(z) ]
        inner_expr = (z - p_exact)**mult * Y_z
        derivative_expr = sp.diff(inner_expr, z, mult - 1)
        residue = sp.limit(derivative_expr, z, p_exact) / sp.factorial(mult - 1)
        term = sp.simplify(residue * u)
        print(f" -> Pole of multiplicity {mult} at z = {p_exact}:")
        
    display(term)
    res_sum += term

# 5. Final Inverse Z-Transform expression with clean expansion
x_n = sp.expand(res_sum)

print("\n5. Final clean inverse Z-transform signal x[n]:")
display(x_n)